# Understanding Tokens, Context, and the Illusion of Memory

## Project Overview
This notebook adapts the Day 4 lesson from the LLM Engineering course into a MapleStory-themed project.

The notebook demonstrates two important LLM concepts:

1. Tokenization - how models process text as tokens rather than words.
2. The Illusion of Memory - why chatbots appear to remember previous conversations despite being stateless.
Rather than using generic examples, this notebook uses MapleStory class changes to show how conversation history creates the appearance of memory.

## Part 1 - Tokenization

### What Are Tokens?
Large Language Models do not read words directly. Instead, text is broken into smaller units called tokens.

Understanding tokens is important because they influence:
- Context window limits
- API costs
- Model behavior
- Prompt design

## Imports

In [2]:
import tiktoken

## Create an Encoding

In [3]:
encoding = tiktoken.encoding_for_model(
    "gpt-4.1-mini"
)

## Encode MapleStory Text

In [ ]:
text = """
My main is Night Lord.
I recently switched to Ren.
"""

tokens = encoding.encode(text)

print(tokens)

## Inspect Individual Tokens

In [ ]:
for token_id in tokens:

    token_text = encoding.decode(
        [token_id]
    )

    print(
        f"{token_id} = {repr(token_text)}"
    )

## Decode Tokens Back Into Text

In [ ]:
decoded_text = encoding.decode(
    tokens
)

print(decoded_text)

## Reflection
This demonstrates that LLMs process token IDs rather than human-readable text.

Even terms such as:
```
Night Lord
Ren
MapleStory
```

may be split into multiple tokens.


# Part 2 - The Illusion of Memory

## Objective
Many users assume that ChatGPT remembers previous conversations.

Let's test that assumption.

## Imports

In [8]:
import os

from dotenv import load_dotenv
from openai import OpenAI

## Initialize Client

In [9]:
load_dotenv(
    override=True
)

client = OpenAI()

## Experiment 1

### Tell the Model Our Main

In [10]:
messages = [
    {
        "role": "system",
        "content":
        "You are a MapleStory assistant."
    },
    {
        "role": "user",
        "content":
        "My MapleStory main is Night Lord."
    }
]

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
)

print(
    response
    .choices[0]
    .message
    .content
)

## Experiment 2

### Ask Again in a New Conversation

In [12]:
messages = [
    {
        "role": "system",
        "content":
        "You are a MapleStory assistant."
    },
    {
        "role": "user",
        "content":
        "What is my MapleStory main?"
    }
]

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
)

print(
    response
    .choices[0]
    .message
    .content
)

Expected:
```
I don't know.
```

## Why Did It Forget?
The answer is simple: LLMs are stateless.

Every API call is independent.

The model never saw:

```
My MapleStory main is Night Lord.
```

during the second request.

## Experiment 3

### Creating the Illusion of Memory
Now we include the conversation history ourselves.

In [14]:
messages = [
    {
        "role": "system",
        "content":
        "You are a MapleStory assistant."
    },
    {
        "role": "user",
        "content":
        "My MapleStory main is Night Lord."
    },
    {
        "role": "assistant",
        "content":
        "Got it. You main Night Lord."
    },
    {
        "role": "user",
        "content":
        "Actually I switched to Ren."
    },
    {
        "role": "assistant",
        "content":
        "Understood. You now main Ren."
    },
    {
        "role": "user",
        "content":
        "What class do I main now?"
    }
]


## Ask the Question

In [15]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
)

print(
    response
    .choices[0]
    .message
    .content
)

You now main Ren.


Expected
```
You currently main Ren.
```

## Visualizing the Context

```
Night Lord
↓
Switched
↓
Ren
↓
Current Main?
```
Because the full conversation is included in the prompt, the model can infer that the correct answer is Ren.

## Key Takeaways

### Tokenization
- LLMs process tokens rather than words
- Context windows are measured in tokens
- Token count affects API cost

### Memory
- LLMs do not remember previous conversations
- Every API call is stateless
- Chatbot memory is created by sending conversation history
- Context management is an application responsibility

## Final Reflection
This project demonstrates one of the most important concepts in LLM engineering: the difference between model capability and application design.

Although the assistant appears to remember that a player switched mains from Night Lord to Ren, *no persistent memory* exists inside the model. The apparent memory is produced by supplying previous messages as context with every request.

Understanding this distinction is essential when building chatbots, assistants, agents, and retrieval-augmented applications because maintaining context is the responsibility of the AI engineer rather than the model itself.